In [ ]:
import pandas as pd
import numpy as np

# Load datasets (same as in eda_aniekan notebook)
fake = pd.read_csv("../data/Fake.csv")
true = pd.read_csv("../data/True.csv")
# new_true1 = pd.read_csv("../data/new_true1.csv")
# new_fake1 = pd.read_csv("../data/new_fake1.csv")

new_fake_and_true = pd.read_csv("../data/Fake_Real_News_Data.csv")




# Add labels (0 = Fake, 1 = Real)
fake["label"] = 0
true["label"] = 1

df = pd.concat([fake, true], ignore_index=True)

# Data cleaning
word_to_remove = '(Reuters)'
df['text'] =  df['text'].str.replace(r'[\(\-–\s]*Reuters[\)\s]*', '', case=False, regex=True)
df['text'] = df['text'].str.replace(r'^[A-Z]+(?:\s+[A-Z]+)*-\s*', '', regex=True)


# Drop Subject
df = df.drop(columns='subject')

df

In [ ]:
new_fake_and_true.head()
new_fake_and_true.tail()

In [ ]:
# new_true1 = pd.read_csv("../data/new_true1.csv")
# new_fake1 = pd.read_csv("../data/new_fake1.csv")

# new_df = pd.concat([new_true1, new_fake1], ignore_index=True)
new_fake_and_true.head()
new_fake_and_true.tail()
new_fake_and_true.info()
new_fake_and_true.describe()
new_fake_and_true

# duplicate = new_fake_and_true.merge(df, how='inner')
# print(f"Number of duplicate rows in new_df: {len(duplicate)}")

In [ ]:

from sklearn.feature_extraction.text import CountVectorizer
faketext = df[df['label'] == 0]['text'].apply(lambda x: ' '.join(x) if isinstance(x, list) else str(x))
realtext = df[df['label'] == 1]['text'].apply(lambda x: ' '.join(x) if isinstance(x, list) else str(x))


# We fit one vectorizer on both fake + real text combined.
# This ensures both datasets share the same vocabulary (columns = same words).
vectorizer = CountVectorizer(stop_words='english')  # remove common filler words like "the", "and"
vectorizer.fit(pd.concat([faketext, realtext]))

# Transform each dataset separately to get their word-count matrices
X_fake_text = vectorizer.transform(faketext)
X_real_text = vectorizer.transform(realtext)

print(vectorizer.get_feature_names_out())

print(X_fake_text)
print(X_real_text)

In [ ]:
# Multinomial Naive Bayes Model trained with only fake and real text data

# Import necessary packages
from scipy.sparse import vstack
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# combining faketext and realtext
X_all = vstack([X_fake_text, X_real_text])
y_all = np.concatenate([
    np.zeros(len(faketext)), #label 0 for fake
    np.ones(len(realtext)) # label 1 for real
])

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.2, random_state=42)

# Train Multinomial Naive Bayes
model = MultinomialNB()
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
new_fake_and_true = pd.read_csv("../data/Fake_Real_News_Data.csv")

if 'Unnamed: 0' in new_fake_and_true.columns:
    new_fake_and_true = new_fake_and_true.drop(columns='Unnamed: 0')

new_fake_and_true.head()

label_map = {'FAKE': 0, 'REAL': 1}
new_fake_and_true['label'] = new_fake_and_true['label'].map(label_map)

new_text = new_fake_and_true['text'].apply(lambda x: ' '.join(x) if isinstance(x, list) else str(x))

X_new = vectorizer.transform(new_text)

y_new = new_fake_and_true['label'].values

y_new_pred = model.predict(X_new)

print(f"Accuracy score on new dataset:", accuracy_score(y_new, y_new_pred))
print(classification_report(y_new, y_new_pred, target_names =['Fake', 'Real']))




In [ ]:
new_fake_and_true.head()


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
welfake_data = pd.read_csv("../data/WELFake_Dataset.csv")

if 'Unnamed: 0' in welfake_data.columns:
    welfake_data = welfake_data.drop(columns='Unnamed: 0')

welfake_data.head()

label_map = {'FAKE': 0, 'REAL': 1}
welfake_data['label'] = welfake_data['label'].map(label_map)
print(welfake_data['label'].isna().sum())
welfake_data = welfake_data.dropna(subset=['label'])


new_text = welfake_data['text'].apply(lambda x: ' '.join(x) if isinstance(x, list) else str(x))

X_new = vectorizer.transform(new_text)

y_new = welfake_data['label'].values

y_new_pred = model.predict(X_new)

print(f"Accuracy score on new dataset:", accuracy_score(y_new, y_new_pred))
print(classification_report(y_new, y_new_pred, target_names =['Fake', 'Real']))




In [ ]:
# Testing Model on WELFake Dataset (with proper error handling)

# Load WELFake dataset
welfake_data = pd.read_csv("../data/WELFake_Dataset.csv")

if 'Unnamed: 0' in welfake_data.columns:
    welfake_data = welfake_data.drop(columns='Unnamed: 0')

print(f"Original dataset shape: {welfake_data.shape}")
print(f"Columns: {welfake_data.columns.tolist()}")

# Check for missing values
print(f"\nMissing values in 'text': {welfake_data['text'].isna().sum()}")
print(f"Missing values in 'title': {welfake_data['title'].isna().sum()}")

# Handle label mapping if needed
if welfake_data['label'].dtype == 'object':
    label_map = {'FAKE': 0, 'REAL': 1, 'fake': 0, 'real': 1, 'Fake': 0, 'Real': 1}
    welfake_data['label'] = welfake_data['label'].map(label_map)

# Prepare text data - handle NaN and empty strings
new_text = welfake_data['text'].apply(
    lambda x: ' '.join(x) if isinstance(x, list) else str(x) if pd.notna(x) else ''
)

# Filter out empty text before transforming
# This is crucial - empty text causes vectorizer to return 0 samples
non_empty_mask = new_text.str.strip().str.len() > 0
welfake_data_filtered = welfake_data[non_empty_mask].copy()
new_text_filtered = new_text[non_empty_mask]

print(f"\nAfter filtering empty text: {len(welfake_data_filtered)} samples")
print(f"Removed {len(welfake_data) - len(welfake_data_filtered)} empty text samples")

if len(new_text_filtered) == 0:
    raise ValueError("All text samples are empty after processing! Check your data cleaning steps.")

# Transform using the trained vectorizer
print("\nTransforming text data using the trained vectorizer...")
X_new = vectorizer.transform(new_text_filtered)

print(f"X_new shape: {X_new.shape}")

if X_new.shape[0] == 0:
    raise ValueError("Vectorizer returned 0 samples. Check if text data is valid.")

# Get labels for filtered data
y_new = welfake_data_filtered['label'].values

# Make predictions
print("Making predictions...")
y_new_pred = model.predict(X_new)

# Evaluate
print(f"\nAccuracy score on WELFake dataset: {accuracy_score(y_new, y_new_pred):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_new, y_new_pred, target_names=['Fake', 'Real']))
